In [0]:
dbutils.widgets.text("catalog", "fk_ce_2026_07_1363")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

In [0]:
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType, DoubleType

df_orders_bronze = spark.table(f"{catalog}.bronze.orders")

# Define items schema as needed for parsing
items_schema = ArrayType(StructType([
    StructField('product_id', StringType()),
    StructField('qty', IntegerType()),
    StructField('unit_price', DoubleType()),
    StructField('discount', DoubleType()),
]))

# Parse JSON string to array of struct
df_orders_bronze_clean = df_orders_bronze.withColumn(
    "items_array", F.from_json(F.col("items"), items_schema)
)

# Explode items for one line per item
df_order_items_tmp = df_orders_bronze_clean.withColumn("item", F.explode("items_array"))

# Select columns for further processing (raw, no cleaning yet)
df_order_items_tmp = df_order_items_tmp.select(
    F.col("order_id"),
    F.col("item.product_id").alias("product_id"),
    F.col("item.qty").alias("quantity"),
    F.col("item.unit_price").alias("unit_price"),
    F.col("item.discount").alias("discount_amount"),
    # The 1-based position within the items array
    (F.expr("ARRAY_POSITION(items_array, item)")).alias("line_number")
)

df_order_items_tmp.display()

# df_order_items_tmp now holds one row per order-item, ready for validation or writing to silver.order_items later.
# All downstream validations and assignments (gross_amount, net_amount, etc) should be added in the table creation code as per silver spec.
